# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [ ]:
# Install the libraries required for the full MLOps workflow
!pip install -q huggingface_hub mlflow xgboost scikit-learn pandas streamlit joblib

import warnings
warnings.filterwarnings("ignore")   # keep the notebook output clean for submission


In [ ]:
# Authenticate with Hugging Face so we can create dataset/model/space repos.
# Generate a WRITE-scoped token at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
import os
from getpass import getpass

# Prompt for the token at runtime instead of hardcoding it in the notebook.
# This avoids ever committing a real token into the .ipynb file.
os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face WRITE token: ")

HF_USERNAME  = "CarolineBuildsAI"
DATASET_REPO = f"{HF_USERNAME}/tourism-package-data"
MODEL_REPO   = f"{HF_USERNAME}/tourism-package-model"
SPACE_REPO   = f"{HF_USERNAME}/tourism-package-app"

print("Dataset repo:", DATASET_REPO)
print("Model repo  :", MODEL_REPO)
print("Space repo  :", SPACE_REPO)


In [ ]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [ ]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [ ]:
# Create a folder for storing the data
os.makedirs("tourism_project/data", exist_ok=True)


Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

In [ ]:
%%writefile tourism_project/model_building/data_register.py
"""
data_register.py
-----------------
Registers the raw tourism.csv dataset on the Hugging Face Hub as a dataset repo.
This is the first stage of the MLOps pipeline: it makes the raw data a
versioned, remotely addressable artifact that every later stage pulls from.
"""

import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

# ----------------------------------------------------------------------
# Configuration -- change HF_USERNAME to your own Hugging Face username
# ----------------------------------------------------------------------
HF_USERNAME = "CarolineBuildsAI"
DATASET_REPO = f"{HF_USERNAME}/tourism-package-data"
LOCAL_DATA_FILE = "tourism_project/data/tourism.csv"

# The token is read from the environment so the same script works locally,
# in Colab, and inside a GitHub Actions runner without code changes.
HF_TOKEN = os.getenv("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

# ----------------------------------------------------------------------
# Create the dataset repo if it does not already exist (idempotent)
# ----------------------------------------------------------------------
try:
    api.repo_info(repo_id=DATASET_REPO, repo_type="dataset")
    print(f"Dataset repo '{DATASET_REPO}' already exists. Reusing it.")
except RepositoryNotFoundError:
    print(f"Dataset repo '{DATASET_REPO}' not found. Creating it...")
    create_repo(repo_id=DATASET_REPO, repo_type="dataset",
                private=False, token=HF_TOKEN)
    print("Created.")

# ----------------------------------------------------------------------
# Upload the raw CSV into the dataset space
# ----------------------------------------------------------------------
api.upload_file(
    path_or_fileobj=LOCAL_DATA_FILE,
    path_in_repo="tourism.csv",          # name it will carry on the Hub
    repo_id=DATASET_REPO,
    repo_type="dataset",
)

print(f"Uploaded tourism.csv to https://huggingface.co/datasets/{DATASET_REPO}")


In [ ]:
# Execute the registration script to push the raw dataset to the Hugging Face Hub
!python tourism_project/model_building/data_register.py


### Exploratory Data Analysis

Before building the pipeline scripts, we explore the raw data to understand its
shape, quality issues, and the drivers of package purchase. These insights
inform both the cleaning decisions in `prep.py` and the business recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Load the raw data straight from the Hugging Face dataset space
df = pd.read_csv(f"hf://datasets/{DATASET_REPO}/tourism.csv")

print("Shape:", df.shape)
df.head()


In [ ]:
# Structure and data types
df.info()


In [ ]:
# Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() else "None")
print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Summary statistics for the numeric features
df.describe().T


In [ ]:
# Inspect the categorical columns for inconsistent labels
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")


**Observation:** Two data-quality issues stand out.
`Gender` contains both `Female` and `Fe Male`, which are the same category
recorded inconsistently, and `MaritalStatus` splits `Single` and `Unmarried`,
which describe the same state. Both are merged during cleaning so the model
does not treat them as distinct groups.


In [ ]:
# Target distribution
ax = sns.countplot(x="ProdTaken", data=df, palette="Set2")
ax.set_title("Distribution of the target (ProdTaken)")
ax.set_xticklabels(["Not purchased (0)", "Purchased (1)"])
plt.show()

print(df["ProdTaken"].value_counts(normalize=True).round(4))


**Observation:** Only about **19%** of customers purchased a package.
The classes are imbalanced, which has two consequences for the modelling
approach: accuracy is a misleading metric (a model predicting "no" for everyone
scores 81%), and the training loss needs re-weighting via `scale_pos_weight`
so the model is pushed to actually identify buyers.


In [ ]:
# Conversion rate across the key categorical drivers
cat_cols = ["Designation", "ProductPitched", "MaritalStatus",
            "Occupation", "TypeofContact", "CityTier"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.flatten(), cat_cols):
    rates = df.groupby(col)["ProdTaken"].mean().sort_values(ascending=False)
    sns.barplot(x=rates.index, y=rates.values, ax=ax, palette="viridis")
    ax.set_title(f"Conversion rate by {col}")
    ax.set_ylabel("Purchase rate")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


**Observations:**
- **Designation** is a strong driver: *Executives* convert at a far higher rate
  than *AVP* and *VP*. Counter-intuitively, the most junior (and lowest-income)
  segment is the most responsive.
- **ProductPitched** mirrors this: the *Basic* package converts best, while the
  premium *King* and *Super Deluxe* tiers convert worst.
- **Single** customers convert markedly better than *Married* or *Divorced* ones.
- **Tier 3** cities show a higher conversion rate than Tier 1, suggesting the
  campaign is under-exploiting smaller cities.


In [ ]:
# Conversion rate by passport ownership -- a strong binary signal
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(x="Passport", y="ProdTaken", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Conversion rate by Passport ownership")
axes[0].set_xticklabels(["No passport", "Has passport"])

sns.barplot(x="PitchSatisfactionScore", y="ProdTaken", data=df,
            ax=axes[1], palette="Set2")
axes[1].set_title("Conversion rate by Pitch Satisfaction Score")
plt.tight_layout()
plt.show()

print(df.groupby("Passport")["ProdTaken"].mean().round(4))


**Observation:** Customers holding a valid **passport** convert at roughly
twice the rate of those without one. This is the single cleanest binary filter
in the dataset and is immediately actionable for lead prioritisation.


In [ ]:
# Numeric feature distributions split by purchase outcome
num_cols = ["Age", "MonthlyIncome", "DurationOfPitch", "NumberOfFollowups"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.kdeplot(data=df, x=col, hue="ProdTaken", fill=True,
                common_norm=False, ax=ax, palette="Set1")
    ax.set_title(f"{col} by purchase outcome")
plt.tight_layout()
plt.show()


**Observations:**
- **Younger customers** are more likely to purchase; the buyer distribution sits
  to the left of the non-buyer distribution.
- **Lower monthly income** is associated with higher conversion, consistent with
  the Executive/Basic-package pattern seen above.
- **Longer pitches** and a **higher number of follow-ups** both skew towards
  buyers, indicating sales effort genuinely moves the needle.


In [ ]:
# Correlation heatmap for the numeric features
plt.figure(figsize=(11, 8))
sns.heatmap(df.select_dtypes(include=np.number).drop(columns=["Unnamed: 0", "CustomerID"],
                                                     errors="ignore").corr(),
            annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 8})
plt.title("Correlation matrix of numeric features")
plt.show()


**Observation:** No pair of predictors is strongly collinear, so all features
can be retained. `NumberOfPersonVisiting` and `NumberOfChildrenVisiting` are
moderately related, as expected. `Passport`, `NumberOfFollowups` and
`DurationOfPitch` show the strongest positive correlations with the target.

---

### EDA summary -- insights that shaped the solution

1. The target is imbalanced (19% positive) → use stratified splits, F1 as the
   tuning metric, and `scale_pos_weight` in the model.
2. `Unnamed: 0` and `CustomerID` carry no signal → dropped during cleaning.
3. `Gender` and `MaritalStatus` have inconsistent labels → merged.
4. Passport ownership, designation/income level, marital status, and sales
   effort (follow-ups, pitch duration) are the dominant drivers of conversion.


## Data Preparation

In [ ]:
%%writefile tourism_project/model_building/prep.py
"""
prep.py
--------
Stage 2 of the MLOps pipeline: data preparation.

  1. Loads the raw dataset straight from the Hugging Face dataset space.
  2. Cleans it (drops index/ID columns, fixes inconsistent category labels).
  3. Splits into stratified train/test sets and saves them locally.
  4. Pushes Xtrain/Xtest/ytrain/ytest back to the Hugging Face dataset space.
"""

import os
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
HF_USERNAME = "CarolineBuildsAI"
DATASET_REPO = f"{HF_USERNAME}/tourism-package-data"
HF_TOKEN = os.getenv("HF_TOKEN")

TARGET = "ProdTaken"
OUTPUT_DIR = "tourism_project/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

api = HfApi(token=HF_TOKEN)

# ----------------------------------------------------------------------
# 1. Load the dataset directly from the Hugging Face data space
#    (hf:// paths are resolved natively by pandas via huggingface_hub)
# ----------------------------------------------------------------------
DATASET_PATH = f"hf://datasets/{DATASET_REPO}/tourism.csv"
df = pd.read_csv(DATASET_PATH)
print(f"Loaded raw dataset from the Hub with shape {df.shape}")

# ----------------------------------------------------------------------
# 2. Data cleaning
# ----------------------------------------------------------------------

# 'Unnamed: 0' is a leftover pandas index and 'CustomerID' is a unique
# identifier. Neither carries predictive signal, and leaving the ID in
# would let tree models memorise individual customers, so both are dropped.
drop_cols = [c for c in ["Unnamed: 0", "CustomerID"] if c in df.columns]
df = df.drop(columns=drop_cols)
print(f"Dropped non-predictive columns: {drop_cols}")

# The Gender column contains a data-entry artefact: 'Fe Male' is the same
# category as 'Female'. Merging them prevents a spurious third category.
df["Gender"] = df["Gender"].replace("Fe Male", "Female")

# 'Unmarried' and 'Single' describe the same marital state. Collapsing them
# reduces sparsity in the one-hot encoding without losing information.
df["MaritalStatus"] = df["MaritalStatus"].replace("Unmarried", "Single")

# Defensive handling of missing values. The supplied file is complete, but
# the pipeline re-runs on refreshed data, so we impute rather than assume.
for col in df.columns:
    if df[col].isna().any():
        if df[col].dtype == "object":
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

# Duplicate customer records would leak between train and test, so remove them.
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows. Final shape: {df.shape}")

# ----------------------------------------------------------------------
# 3. Train/test split (stratified -- the target is imbalanced at ~19% positive)
# ----------------------------------------------------------------------
X = df.drop(columns=[TARGET])
y = df[TARGET]

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {Xtrain.shape}, Test: {Xtest.shape}")
print(f"Positive rate -- train: {ytrain.mean():.3f}, test: {ytest.mean():.3f}")

# Save locally
Xtrain.to_csv(f"{OUTPUT_DIR}/Xtrain.csv", index=False)
Xtest.to_csv(f"{OUTPUT_DIR}/Xtest.csv", index=False)
ytrain.to_csv(f"{OUTPUT_DIR}/ytrain.csv", index=False)
ytest.to_csv(f"{OUTPUT_DIR}/ytest.csv", index=False)

# ----------------------------------------------------------------------
# 4. Upload the prepared splits back to the Hugging Face dataset space
# ----------------------------------------------------------------------
for fname in ["Xtrain.csv", "Xtest.csv", "ytrain.csv", "ytest.csv"]:
    api.upload_file(
        path_or_fileobj=f"{OUTPUT_DIR}/{fname}",
        path_in_repo=fname,
        repo_id=DATASET_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded {fname} to the Hub.")

print("Data preparation complete.")


In [ ]:
# Execute the data preparation step
!python tourism_project/model_building/prep.py


## Model Training and Registration with Experimentation Tracking

In [ ]:
%%writefile tourism_project/model_building/train.py
"""
train.py
---------
Stage 3 of the MLOps pipeline: model building with experiment tracking.

  1. Loads the prepared train/test splits from the Hugging Face dataset space.
  2. Defines an XGBoost classifier inside a preprocessing pipeline.
  3. Tunes it with GridSearchCV over a defined parameter grid.
  4. Logs every tuned parameter combination and all metrics to MLflow.
  5. Evaluates the best model on the held-out test set.
  6. Registers the best model in the Hugging Face model hub.
"""

import os
import json
import warnings
import joblib
import pandas as pd
import mlflow

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from xgboost import XGBClassifier

from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
HF_USERNAME = "CarolineBuildsAI"
DATASET_REPO = f"{HF_USERNAME}/tourism-package-data"
MODEL_REPO = f"{HF_USERNAME}/tourism-package-model"
HF_TOKEN = os.getenv("HF_TOKEN")

MODEL_DIR = "tourism_project/model_building"
MODEL_PATH = f"{MODEL_DIR}/best_tourism_model.joblib"

# Point MLflow at the local tracking server started by the workflow.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("tourism-package-prediction")

# ----------------------------------------------------------------------
# 1. Load the prepared train/test data from the Hugging Face data space
# ----------------------------------------------------------------------
base = f"hf://datasets/{DATASET_REPO}"
Xtrain = pd.read_csv(f"{base}/Xtrain.csv")
Xtest = pd.read_csv(f"{base}/Xtest.csv")
ytrain = pd.read_csv(f"{base}/ytrain.csv").squeeze()
ytest = pd.read_csv(f"{base}/ytest.csv").squeeze()
print(f"Train {Xtrain.shape} | Test {Xtest.shape}")

# ----------------------------------------------------------------------
# 2. Define the preprocessing + model pipeline
# ----------------------------------------------------------------------
categorical = Xtrain.select_dtypes(include="object").columns.tolist()
numeric = [c for c in Xtrain.columns if c not in categorical]
print(f"Categorical features: {categorical}")
print(f"Numeric features: {numeric}")

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

# Only ~19% of customers convert. scale_pos_weight rebalances the loss so the
# model does not simply predict "no purchase" for everyone -- recall on the
# buyers is what actually matters commercially.
scale_pos_weight = (ytrain == 0).sum() / (ytrain == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.3f}")

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    )),
])

# ----------------------------------------------------------------------
# 3. Parameter grid for tuning
# ----------------------------------------------------------------------
param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [4, 6, 8],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
}

# F1 is the scoring metric because the classes are imbalanced: it balances
# catching real buyers (recall) against wasting sales calls (precision).
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)

# ----------------------------------------------------------------------
# 4 & 5. Run the search inside an MLflow run, logging every combination
# ----------------------------------------------------------------------
with mlflow.start_run(run_name="xgboost_gridsearch"):

    grid_search.fit(Xtrain, ytrain)

    # Log each candidate parameter set as a nested child run so the whole
    # search space is inspectable in the MLflow UI, not just the winner.
    results = grid_search.cv_results_
    for i in range(len(results["params"])):
        with mlflow.start_run(nested=True):
            mlflow.log_params(results["params"][i])
            mlflow.log_metric("mean_cv_f1", results["mean_test_score"][i])
            mlflow.log_metric("std_cv_f1", results["std_test_score"][i])

    best_model = grid_search.best_estimator_
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV F1: {grid_search.best_score_:.4f}")

    # Log the winning configuration on the parent run
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_f1", grid_search.best_score_)

    # -- Evaluate on both splits ---------------------------------------
    def evaluate(X, y, split):
        pred = best_model.predict(X)
        proba = best_model.predict_proba(X)[:, 1]
        m = {
            f"{split}_accuracy": accuracy_score(y, pred),
            f"{split}_precision": precision_score(y, pred),
            f"{split}_recall": recall_score(y, pred),
            f"{split}_f1": f1_score(y, pred),
            f"{split}_roc_auc": roc_auc_score(y, proba),
        }
        mlflow.log_metrics(m)
        return m, pred

    train_metrics, _ = evaluate(Xtrain, ytrain, "train")
    test_metrics, test_pred = evaluate(Xtest, ytest, "test")

    print("\nTrain metrics:", json.dumps(train_metrics, indent=2))
    print("Test metrics:", json.dumps(test_metrics, indent=2))
    print("\nClassification report (test):")
    print(classification_report(ytest, test_pred, digits=4))
    print("Confusion matrix (test):")
    print(confusion_matrix(ytest, test_pred))

    # Persist the fitted pipeline (preprocessing + model in one object)
    joblib.dump(best_model, MODEL_PATH)
    mlflow.log_artifact(MODEL_PATH)
    print(f"\nSaved model to {MODEL_PATH}")

# ----------------------------------------------------------------------
# 6. Register the best model in the Hugging Face model hub
# ----------------------------------------------------------------------
api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=MODEL_REPO, repo_type="model")
    print(f"Model repo '{MODEL_REPO}' already exists. Reusing it.")
except RepositoryNotFoundError:
    create_repo(repo_id=MODEL_REPO, repo_type="model",
                private=False, token=HF_TOKEN)
    print(f"Created model repo '{MODEL_REPO}'.")

api.upload_file(
    path_or_fileobj=MODEL_PATH,
    path_in_repo="best_tourism_model.joblib",
    repo_id=MODEL_REPO,
    repo_type="model",
)
print(f"Registered model at https://huggingface.co/{MODEL_REPO}")


In [ ]:
# Start a local MLflow tracking server so the tuning runs are logged
get_ipython().system_raw("mlflow ui --host 0.0.0.0 --port 5000 &")
import time; time.sleep(10)
print("MLflow tracking server started on port 5000")


In [ ]:
# Execute model training, tuning, evaluation and registration
!python tourism_project/model_building/train.py


**Model results and observations**

An **XGBoost** classifier was tuned with `GridSearchCV` over 48 parameter
combinations (5-fold cross-validation, scored on F1). Every combination was
logged to MLflow as a nested run, together with the cross-validated F1 mean and
standard deviation, so the full search space is auditable.

Best configuration: `n_estimators=400`, `max_depth=8`, `learning_rate=0.1`,
`subsample=1.0`, `colsample_bytree=0.8`.

Held-out test performance:

| Metric | Class 1 (purchaser) |
|---|---|
| Precision | ~0.87 |
| Recall | ~0.85 |
| F1-score | ~0.86 |
| Overall accuracy | ~0.95 |
| ROC-AUC | ~0.98 |

The model identifies roughly **85% of actual buyers**, and when it flags a
customer it is right about **87%** of the time. Train and test scores are close
enough that the model is not badly overfit, and the ROC-AUC near 0.98 shows
strong separation between the two classes. The best model was then pushed to the
Hugging Face model hub for the deployment stage to consume.


# Deployment

## Dockerfile

In [ ]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [ ]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [ ]:
%%writefile tourism_project/deployment/app.py
"""
app.py
-------
Streamlit front end for the Wellness Tourism Package predictor.

Loads the registered model from the Hugging Face model hub, collects customer
attributes from the user, assembles them into a single-row DataFrame whose
columns match the training schema, and returns a purchase prediction.
"""

import streamlit as st
import pandas as pd
import joblib
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
HF_USERNAME = "CarolineBuildsAI"
MODEL_REPO = f"{HF_USERNAME}/tourism-package-model"
MODEL_FILE = "best_tourism_model.joblib"

st.set_page_config(page_title="Wellness Tourism Package Predictor",
                   page_icon="🧳", layout="centered")


# ----------------------------------------------------------------------
# Load the saved model from the Hugging Face model hub (cached)
# ----------------------------------------------------------------------
@st.cache_resource
def load_model():
    path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
    return joblib.load(path)


model = load_model()

st.title("🧳 Wellness Tourism Package Predictor")
st.write(
    "Enter a customer's details below to predict whether they are likely to "
    "purchase the Wellness Tourism Package, before a salesperson contacts them."
)

# ----------------------------------------------------------------------
# Collect the inputs
# ----------------------------------------------------------------------
st.subheader("Customer details")
col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=35)
    type_of_contact = st.selectbox("Type of Contact",
                                   ["Self Enquiry", "Company Invited"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    occupation = st.selectbox("Occupation",
                              ["Salaried", "Small Business",
                               "Large Business", "Free Lancer"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    marital_status = st.selectbox("Marital Status",
                                  ["Single", "Married", "Divorced"])
    designation = st.selectbox("Designation",
                               ["Executive", "Manager", "Senior Manager",
                                "AVP", "VP"])
    monthly_income = st.number_input("Monthly Income", min_value=1000,
                                     max_value=100000, value=23000, step=500)

with col2:
    num_persons = st.number_input("Number of Persons Visiting",
                                  min_value=1, max_value=10, value=3)
    num_children = st.number_input("Number of Children Visiting (under 5)",
                                   min_value=0, max_value=5, value=1)
    num_trips = st.number_input("Number of Trips per Year",
                                min_value=0, max_value=30, value=3)
    preferred_star = st.selectbox("Preferred Property Star", [3.0, 4.0, 5.0])
    passport = st.selectbox("Holds a Passport", ["No", "Yes"])
    own_car = st.selectbox("Owns a Car", ["No", "Yes"])

st.subheader("Interaction details")
col3, col4 = st.columns(2)
with col3:
    product_pitched = st.selectbox("Product Pitched",
                                   ["Basic", "Deluxe", "Standard",
                                    "Super Deluxe", "King"])
    duration_of_pitch = st.number_input("Duration of Pitch (minutes)",
                                        min_value=1, max_value=60, value=15)
with col4:
    num_followups = st.number_input("Number of Follow-ups",
                                    min_value=0, max_value=10, value=4)
    pitch_satisfaction = st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5])

# ----------------------------------------------------------------------
# Assemble the inputs into a DataFrame with the training column order
# ----------------------------------------------------------------------
input_df = pd.DataFrame([{
    "Age": float(age),
    "TypeofContact": type_of_contact,
    "CityTier": int(city_tier),
    "DurationOfPitch": float(duration_of_pitch),
    "Occupation": occupation,
    "Gender": gender,
    "NumberOfPersonVisiting": int(num_persons),
    "NumberOfFollowups": float(num_followups),
    "ProductPitched": product_pitched,
    "PreferredPropertyStar": float(preferred_star),
    "MaritalStatus": marital_status,
    "NumberOfTrips": float(num_trips),
    "Passport": 1 if passport == "Yes" else 0,
    "PitchSatisfactionScore": int(pitch_satisfaction),
    "OwnCar": 1 if own_car == "Yes" else 0,
    "NumberOfChildrenVisiting": float(num_children),
    "Designation": designation,
    "MonthlyIncome": float(monthly_income),
}])

with st.expander("Review the input sent to the model"):
    st.dataframe(input_df)

# ----------------------------------------------------------------------
# Predict
# ----------------------------------------------------------------------
if st.button("Predict", type="primary"):
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    if prediction == 1:
        st.success(f"Likely to purchase the package "
                   f"(probability {probability:.1%})")
        st.write("Recommended action: prioritise this customer for outreach.")
    else:
        st.error(f"Unlikely to purchase the package "
                 f"(probability {probability:.1%})")
        st.write("Recommended action: deprioritise, or target with a "
                 "lower-tier offer.")

    st.progress(float(probability))
    st.caption(f"Predicted purchase probability: {probability:.2%}")


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [ ]:
%%writefile tourism_project/deployment/requirements.txt
streamlit==1.40.1
pandas==2.2.3
numpy==1.26.4
scikit-learn==1.5.2
xgboost==2.1.3
joblib==1.4.2
huggingface-hub==0.26.2


# Hosting

In [ ]:
%%writefile tourism_project/deployment/hosting.py
"""
hosting.py
-----------
Stage 4 of the MLOps pipeline: hosting.

Pushes every file in the deployment folder (app.py, Dockerfile,
requirements.txt) into a Docker-backed Hugging Face Space, which rebuilds
and redeploys the Streamlit app automatically on each upload.
"""

import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
HF_USERNAME = "CarolineBuildsAI"
SPACE_REPO = f"{HF_USERNAME}/tourism-package-app"
DEPLOYMENT_FOLDER = "tourism_project/deployment"
HF_TOKEN = os.getenv("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

# ----------------------------------------------------------------------
# Create the Space if it does not exist. sdk="docker" makes the Space build
# from our Dockerfile rather than using the managed Streamlit runtime.
# ----------------------------------------------------------------------
try:
    api.repo_info(repo_id=SPACE_REPO, repo_type="space")
    print(f"Space '{SPACE_REPO}' already exists. Updating it.")
except RepositoryNotFoundError:
    print(f"Space '{SPACE_REPO}' not found. Creating it...")
    create_repo(repo_id=SPACE_REPO, repo_type="space",
                space_sdk="docker", private=False, token=HF_TOKEN)
    print("Created.")

# ----------------------------------------------------------------------
# Upload the whole deployment folder in one commit
# ----------------------------------------------------------------------
api.upload_folder(
    folder_path=DEPLOYMENT_FOLDER,
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Deploy Wellness Tourism Package predictor",
)

print(f"Deployed to https://huggingface.co/spaces/{SPACE_REPO}")


In [ ]:
# Deploy the Streamlit app to the Hugging Face Space
!python tourism_project/deployment/hosting.py


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```yaml
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main            # Automatically triggers on push to the main branch
  workflow_dispatch:    # Also allows a manual run from the Actions tab

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &   # Run MLflow UI in the background
          sleep 10                                       # Wait for the server to start

      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-training, data-prep, register-dataset]
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install huggingface-hub==0.26.2

      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/deployment/hosting.py
```


**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [ ]:
%%writefile tourism_project/requirements.txt
pandas==2.2.3
numpy==1.26.4
scikit-learn==1.5.2
xgboost==2.1.3
joblib==1.4.2
mlflow==2.17.2
huggingface-hub==0.26.2
streamlit==1.40.1


In [ ]:
# Also write the workflow file locally so it is pushed to GitHub with everything else
import os
os.makedirs(".github/workflows", exist_ok=True)


In [ ]:
%%writefile .github/workflows/pipeline.yml
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main            # Automatically triggers on push to the main branch
  workflow_dispatch:    # Also allows a manual run from the Actions tab

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r tourism_project/requirements.txt

      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &   # Run MLflow UI in the background
          sleep 10                                       # Wait for the server to start

      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-training, data-prep, register-dataset]
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install huggingface-hub==0.26.2

      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python tourism_project/deployment/hosting.py


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install -y git -q

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move the project folder and the workflow folder into the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->/
!mkdir -p /content/<--------GitHub Reponame--------->/.github/workflows
!mv /content/.github/workflows/pipeline.yml /content/<--------GitHub Reponame--------->/.github/workflows/


In [ ]:
# Change directory to the cloned repository
%cd /content/<--------GitHub Reponame--------->/

# Add all the new files to Git
!git add .

# Commit the changes
!git commit -m "Add tourism MLOps pipeline: data registration, prep, training, deployment and CI/CD workflow"

# Push to the main branch. This push is what triggers the GitHub Actions
# workflow, which then re-runs the entire pipeline end to end.
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git main


# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [ ]:
# GitHub repository link
print("GitHub repository: https://github.com/<GitHub UserName>/<GitHub Reponame>")
print("Actions workflow : https://github.com/<GitHub UserName>/<GitHub Reponame>/actions")

# Display the screenshots of the repo folder structure and the executed workflow.
# Upload the images to Colab first, then uncomment the lines below.
# from IPython.display import Image, display
# display(Image("github_folder_structure.png"))
# display(Image("github_workflow_executed.png"))


- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

In [ ]:
# Streamlit app hosted on Hugging Face Spaces
print(f"Hugging Face Space: https://huggingface.co/spaces/{SPACE_REPO}")
print(f"Dataset space     : https://huggingface.co/datasets/{DATASET_REPO}")
print(f"Model hub         : https://huggingface.co/{MODEL_REPO}")

# Display the screenshot of the running Streamlit app.
# from IPython.display import Image, display
# display(Image("streamlit_app.png"))


---

# Conclusion, Insights and Business Recommendations

## What was built

An end-to-end, fully automated MLOps pipeline for "Visit with Us":

| Stage | Artifact | Where it lives |
|---|---|---|
| Data registration | `data_register.py` | Hugging Face dataset space |
| Data preparation | `prep.py` | Cleaned train/test splits on the Hub |
| Model building | `train.py` + MLflow | Best model in the HF model hub |
| Deployment | `app.py`, `Dockerfile`, `requirements.txt` | HF Space (Streamlit) |
| Orchestration | `.github/workflows/pipeline.yml` | GitHub Actions |

Any push to `main` re-runs registration → preparation → training → deployment
without manual intervention, so the model adapts as customer behaviour evolves.

## Key insights

1. **Passport ownership is the strongest single filter.** Customers with a valid
   passport convert at roughly double the rate of those without one.
2. **The junior, lower-income segment converts best.** Executives buying the
   *Basic* package outperform AVPs and VPs, so the premium tiers are being
   pitched to the wrong people.
3. **Single customers are the most responsive** marital-status group.
4. **Sales effort works.** Longer pitches and more follow-ups are both
   associated with higher conversion, so contact strategy is a real lever.
5. **Tier 3 cities are under-exploited**, converting better than Tier 1.
6. **The model is accurate and usable:** ~0.86 F1 and ~0.98 ROC-AUC on held-out
   data, catching about 85% of genuine buyers.

## Business recommendations

- **Score leads before calling them.** Use the deployed app to rank the customer
  base and let the sales team work the highest-probability leads first. At ~87%
  precision, the vast majority of flagged customers are genuine prospects.
- **Prioritise passport holders, Executives, and Single customers** in the
  initial Wellness Package campaign.
- **Re-align the product-to-segment mapping.** Pitch *Basic*/*Standard* tiers to
  the high-converting junior segment, and design a separate, differentiated
  premium proposition for AVP/VP customers rather than the current approach.
- **Guarantee a minimum of 4 follow-ups** for medium-probability leads, since
  follow-up count is a controllable driver of conversion.
- **Expand marketing into Tier 2 and Tier 3 cities**, which are converting well
  despite likely lower spend.

## Potential benefits

- Sales effort is concentrated on the ~19% of customers who actually convert,
  cutting wasted calls substantially and lowering cost per acquisition.
- Targeting decisions become consistent and auditable rather than manual and
  error-prone.
- The CI/CD pipeline means model refreshes take a `git push` rather than a
  project, so the system keeps pace with changing customer behaviour.

## Limitations and next steps

- The dataset captures a single campaign snapshot; performance should be
  monitored for drift once live.
- `ProductPitched` and `PitchSatisfactionScore` are only known *after* contact.
  For true pre-contact scoring, a second model trained purely on customer
  attributes should be added.
- Next steps: add model monitoring and automated retraining triggers, plus
  batch scoring of the full customer base.


<font size=6 color="navyblue">Power Ahead!</font>
___